In [8]:
%load_ext autoreload
%autoreload 2

from MnistClassifier_module import MnistClassifier
from tensorflow.keras.datasets import mnist
from sklearn.metrics import accuracy_score
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
(X_train_raw, y_train), (X_test_raw, y_test) = mnist.load_data()
X_train = X_train_raw / 255
X_test = X_test_raw / 255

### CNN train/test script

In [ ]:
CNN_classifier = MnistClassifier('cnn')
CNN_classifier.train(X_train, y_train)

In [4]:
CNN_prediction = CNN_classifier.predict(X_test)
print('CNN Accuracy: ', accuracy_score(y_test, CNN_prediction))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
Prediction complete
CNN Accuracy:  0.9896


### FFNN train/test script

In [ ]:
FFNN_classifier = MnistClassifier('nn')
FFNN_classifier.train(X_train, y_train)

In [5]:
FFNN_prediction = FFNN_classifier.predict(X_test)
print('FFNN Accuracy: ', accuracy_score(y_test, FFNN_prediction))

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Prediction complete
FFNN Accuracy:  0.9803


### Random Forest train/test script

In [6]:
RF_classifier = MnistClassifier('rf')
RF_classifier.train(X_train, y_train)

Training complete


In [7]:
RF_prediction = RF_classifier.predict(X_test)
print('RF Accuracy: ', accuracy_score(y_test, RF_prediction))

Prediction complete
RF Accuracy:  0.9704


### EDGE CASES DESCRIPTION

* Unexpected non-(n_samples,28,28) input shape will raise ValueError during reshaping or flatten steps

* Unnormalized 0-255 pixel values data instead expected 0-1 scaled pixel values leads to extreme accuracy degradations, especially for NN

* Full of zeros or ones picture data (full black/white picture) doesn't break the code, but make model to predict with very low confidence what is standard result for out-of-distribution data.

* Color inversion is a case of out-of-distribution data. A large number of neural connections with small weights responsible for the background receive signal 1 unlike 0, it leads to highly confident wrong prediction by accident activation of some NN output.

* Nan values on image: in theory, a single NaN pixel should ruin the math and force the model to output an array of NaNs. However, in practice, frameworks like TensorFlow often silently convert these invalid values into zeros. Because of this, the code doesn't crash, but the network loses the signal and simply makes a random, low-confidence guess.

In [61]:
print("Edge Case Demonstration: Incorrect Input Shape")
bad_shape_image = np.zeros((1, 32, 32))

try:
    print("Attempting to predict on shape (1, 32, 32)...")
    RF_classifier.predict(bad_shape_image)
except ValueError as e:
    print(f"Caught expected ValueError: {e}")
    print("System behaves correctly by rejecting invalid input shapes.")

Edge Case Demonstration: Incorrect Input Shape
Attempting to predict on shape (1, 32, 32)...
Caught expected ValueError: X has 1024 features, but RandomForestClassifier is expecting 784 features as input.
System behaves correctly by rejecting invalid input shapes.


In [62]:
print("Edge Case Demonstration: Completely Black Image")
black_image = np.zeros((1, 28, 28))

forced_prediction = FFNN_classifier.predict(black_image)
print(f"Standard predict output (forced guess): {forced_prediction[0]}")

print("Extracting raw probabilities:")
raw_probabilities = FFNN_classifier.model.model.predict(black_image, verbose=0)[0]


for i, prob in enumerate(raw_probabilities):
    print(f"Class {i}: {prob*100:.2f}%")

print(f"Model is predicting {forced_prediction[0]}, but as we see in raw probabilities there are no even near 50% probability class, this indicates high uncertainty and an unreliable prediction")



Edge Case Demonstration: Completely Black Image
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Prediction complete
Standard predict output (forced guess): 5
Extracting raw probabilities:
Class 0: 5.48%
Class 1: 7.14%
Class 2: 12.86%
Class 3: 5.30%
Class 4: 9.35%
Class 5: 22.75%
Class 6: 9.92%
Class 7: 9.67%
Class 8: 10.72%
Class 9: 6.81%
Model is predicting 5, but as we see in raw probabilities there are no even near 50% probability class, this indicates high uncertainty and an unreliable prediction


In [63]:
print("Edge Case Demonstration: Nan value in Image")
nan_image = np.zeros((1, 28, 28))
nan_image[0, 14, 14] = np.nan #single nan in the center of image

nan_prediction = FFNN_classifier.model.model.predict(nan_image, verbose=0)[0]
for i, prob in enumerate(nan_prediction):
    print(f"Class {i}: {prob*100:.2f}%")

print("As we see result is not the same with all-zero image, it indicates that NN is working with NaN value unlike simply replace it by 0.0")




Edge Case Demonstration: Nan value in Image
Class 0: 8.48%
Class 1: 8.20%
Class 2: 9.75%
Class 3: 8.85%
Class 4: 11.18%
Class 5: 10.10%
Class 6: 9.71%
Class 7: 9.20%
Class 8: 14.34%
Class 9: 10.18%
As we see result is not the same with all-zero image, it indicates that NN is working with NaN value unlike simply replace it by 0.0


In [68]:
print("Edge Case Demonstration: Color inverse in Image")
inverted_image = 1 - X_test[0:1]
invert_prediction = FFNN_classifier.model.model.predict(inverted_image, verbose=0)[0]
for i, prob in enumerate(invert_prediction):
    print(f"Class {i}: {prob*100:.2f}%")

print("Unlike the black image, the color-inverted image causes confident failure. The massive number of newly activated background pixels randomly triggers a massive response for a specific class with 99.9% confidence. This shows that out-of-distribution data doesn't always cause uncertainty; it can cause highly confident wrong predictions.")


Edge Case Demonstration: Color inverse in Image
Class 0: 0.00%
Class 1: 0.00%
Class 2: 0.00%
Class 3: 0.00%
Class 4: 0.00%
Class 5: 0.00%
Class 6: 99.98%
Class 7: 0.00%
Class 8: 0.02%
Class 9: 0.00%
As we see result is familiar to other out-of-distribution case with one-colored image
